# Universidad del Valle de Guatemala

## Departamento de Ciencias de la Computacion

## Inteligencia Artificial - Seccion 10



# Laboratorio No. 6



### Nadissa Vela - 23764

### Cristian Tunchez - 231359



## Task 2 - Implementacion Connect Four (4 en linea)



### Introduccion

En este laboratorio implementamos una IA para **Connect Four** utilizando **Minimax** con profundidad limitada.



Connect Four es un juego de suma cero en un tablero de **7 columnas x 6 filas**, donde gana quien conecta 4 fichas consecutivas (horizontal, vertical o diagonal).



La utilidad en estados terminales se define como:

- **+1000** si gana la IA.

- **-1000** si gana el oponente.

- **0** si hay empate.



En **Task 2.1** se implementa:

- La clase `Connect4` para modelar estado, acciones y terminalidad.

- El agente **Minimax recursivo** mediante `get_best_move(board, depth)`.

- Restriccion de profundidad recomendada: **d = 3** o **d = 4**.


## 1. Importaciones y constantes del entorno



En esta seccion se importan utilidades basicas y se definen constantes globales:

- Dimensiones del tablero (`ROWS`, `COLS`).

- Representacion de casillas vacias (`EMPTY`).

- Identificadores de jugadores (`PLAYER_1`, `PLAYER_2`).



Estas constantes se reutilizan en toda la implementacion para mantener consistencia del modelo.

In [ ]:
from copy import deepcopy
import random

ROWS = 6
COLS = 7
EMPTY = 0
PLAYER_1 = 1
PLAYER_2 = 2

## 2. Clase Connect4: modelado del juego



Esta celda implementa la logica principal del entorno de juego:

- `actions(s)`: columnas validas para jugar.

- `make_move(col, player)`: aplica una jugada respetando gravedad.

- `succ(s, a)`: genera estado sucesor.

- `check_winner(player)`: detecta 4 en linea horizontal, vertical y diagonal.

- `is_terminal(s)`: determina fin del juego por victoria o empate.



Con esto se define formalmente el problema como juego de dos jugadores y suma cero.

In [ ]:
class Connect4:
    def __init__(self, board=None):
        if board is None:
            self.board = [[EMPTY for _ in range(COLS)] for _ in range(ROWS)]
        else:
            self.board = deepcopy(board)

    def copy(self):
        return Connect4(self.board)

    def actions(self):
        # Columnas validas: la celda superior aun esta vacia.
        return [c for c in range(COLS) if self.board[0][c] == EMPTY]

    def make_move(self, col, player):
        if col not in self.actions():
            return False
        for r in range(ROWS - 1, -1, -1):
            if self.board[r][col] == EMPTY:
                self.board[r][col] = player
                return True
        return False

    def succ(self, action, player):
        next_state = self.copy()
        next_state.make_move(action, player)
        return next_state

    def check_winner(self, player):
        b = self.board

        # Horizontal
        for r in range(ROWS):
            for c in range(COLS - 3):
                if b[r][c] == player and b[r][c+1] == player and b[r][c+2] == player and b[r][c+3] == player:
                    return True

        # Vertical
        for r in range(ROWS - 3):
            for c in range(COLS):
                if b[r][c] == player and b[r+1][c] == player and b[r+2][c] == player and b[r+3][c] == player:
                    return True

        # Diagonal descendente\
 (\)
        for r in range(ROWS - 3):
            for c in range(COLS - 3):
                if b[r][c] == player and b[r+1][c+1] == player and b[r+2][c+2] == player and b[r+3][c+3] == player:
                    return True

        # Diagonal ascendente (/ )
        for r in range(3, ROWS):
            for c in range(COLS - 3):
                if b[r][c] == player and b[r-1][c+1] == player and b[r-2][c+2] == player and b[r-3][c+3] == player:
                    return True

        return False

    def is_terminal(self):
        return self.check_winner(PLAYER_1) or self.check_winner(PLAYER_2) or len(self.actions()) == 0

    def winner(self):
        if self.check_winner(PLAYER_1):
            return PLAYER_1
        if self.check_winner(PLAYER_2):
            return PLAYER_2
        return None

    def print_board(self):
        # Muestra fila superior primero.
        for r in range(ROWS):
            print(self.board[r])
        print('-' * 25)

## 3. Funcion de evaluacion heuristica



Como Minimax se limita a profundidad `d`, no siempre llega a un estado terminal.

Por eso se implementa `evaluate_board(state, ai_player)`, que aproxima que tan bueno es un estado intermedio.



Criterios principales:

- Priorizar la columna central.

- Recompensar ventanas de 4 con 2 o 3 fichas propias.

- Penalizar amenazas del oponente (3 en linea con un espacio).

In [ ]:
def evaluate_window(window, ai_player):
    opponent = PLAYER_1 if ai_player == PLAYER_2 else PLAYER_2

    if window.count(ai_player) == 4:
        return 100
    if window.count(ai_player) == 3 and window.count(EMPTY) == 1:
        return 5
    if window.count(ai_player) == 2 and window.count(EMPTY) == 2:
        return 2

    if window.count(opponent) == 3 and window.count(EMPTY) == 1:
        return -4

    return 0


def evaluate_board(state, ai_player):
    # Evalua estado no terminal cuando depth == 0.
    score = 0
    b = state.board

    # Preferencia por columna central.
    center_col = COLS // 2
    center_values = [b[r][center_col] for r in range(ROWS)]
    score += center_values.count(ai_player) * 3

    # Horizontal
    for r in range(ROWS):
        row_vals = b[r]
        for c in range(COLS - 3):
            score += evaluate_window(row_vals[c:c+4], ai_player)

    # Vertical
    for c in range(COLS):
        col_vals = [b[r][c] for r in range(ROWS)]
        for r in range(ROWS - 3):
            score += evaluate_window(col_vals[r:r+4], ai_player)

    # Diagonal descendente (\)
    for r in range(ROWS - 3):
        for c in range(COLS - 3):
            score += evaluate_window([b[r+i][c+i] for i in range(4)], ai_player)

    # Diagonal ascendente (/ )
    for r in range(3, ROWS):
        for c in range(COLS - 3):
            score += evaluate_window([b[r-i][c+i] for i in range(4)], ai_player)

    return score

## 4. Agente Minimax y seleccion de jugada



Aqui se implementa el algoritmo **Minimax recursivo**:

- Nodo MAX: turno de la IA (maximiza valor).

- Nodo MIN: turno del oponente (minimiza valor).

- Casos base: estado terminal o profundidad 0.



La funcion clave solicitada en el task es:

- `get_best_move(board, depth, ai_player)`



Para este laboratorio se recomienda `depth=3` o `depth=4` para controlar el tiempo de ejecucion sin poda alfa-beta.

In [ ]:
def minimax(state, depth, maximizing_player, ai_player):
    opponent = PLAYER_1 if ai_player == PLAYER_2 else PLAYER_2

    if state.is_terminal():
        winner = state.winner()
        if winner == ai_player:
            return None, 1000
        if winner == opponent:
            return None, -1000
        return None, 0

    if depth == 0:
        return None, evaluate_board(state, ai_player)

    valid_moves = state.actions()

    if maximizing_player:
        best_score = float('-inf')
        best_move = random.choice(valid_moves)

        for col in valid_moves:
            child = state.succ(col, ai_player)
            _, score = minimax(child, depth - 1, False, ai_player)
            if score > best_score:
                best_score = score
                best_move = col

        return best_move, best_score

    best_score = float('inf')
    best_move = random.choice(valid_moves)

    for col in valid_moves:
        child = state.succ(col, opponent)
        _, score = minimax(child, depth - 1, True, ai_player)
        if score < best_score:
            best_score = score
            best_move = col

    return best_move, best_score


def get_best_move(board, depth=4, ai_player=PLAYER_1):
    """
    board: matriz 6x7 (listas de listas) o instancia Connect4
    depth: recomendado 3 o 4 para Task 2.1
    """
    state = board if isinstance(board, Connect4) else Connect4(board)
    move, _ = minimax(state, depth, True, ai_player)
    return move

## 5. Simulacion de validacion



Esta seccion ejecuta una prueba rapida de funcionamiento:

- IA con Minimax contra un oponente aleatorio.

- Se imprime el tablero final y el ganador.



Objetivo: verificar que la logica del juego y la eleccion de jugadas del agente funcionan correctamente.

In [ ]:
# Prueba rapida: IA vs jugador aleatorio
def play_ai_vs_random(depth=4, ai_player=PLAYER_1, seed=7):
    random.seed(seed)
    game = Connect4()
    current = PLAYER_1

    while not game.is_terminal():
        if current == ai_player:
            col = get_best_move(game, depth=depth, ai_player=ai_player)
        else:
            col = random.choice(game.actions())

        game.make_move(col, current)
        current = PLAYER_1 if current == PLAYER_2 else PLAYER_2

    game.print_board()
    w = game.winner()
    if w is None:
        return "Empate"
    return f"Gana jugador {w}"


resultado = play_ai_vs_random(depth=4, ai_player=PLAYER_1, seed=11)
print("Resultado:", resultado)